# Stage 3 · LLaDA-8B-Instruct: scoring-code tests

Runtime: **A100** (the saved LLaDA run used an A100 with torch 2.6.0; matching it keeps BF16 scores comparable). One fresh runtime for this notebook only.

LLaDA already has a saved sanity run (AUROC 0.885, fails the 0.90 bar). This notebook runs what is still missing: the parity test against the official code (test 2), batch invariance (test 3), the sanity check through the shared scorer, and agreement with the saved run.

In [ ]:
BRANCH = "main"   # the branch to test; main once the stage-3 PR is merged
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r harness/diffusion/requirements/llada.txt
!git clone -q https://github.com/ML-GSAI/LLaDA /content/LLaDA && git -C /content/LLaDA checkout -q 9182493
OUT = "results/stage3/llada"
!mkdir -p {OUT}
import hashlib
DEV_HASH = hashlib.sha256(open("bank/dev/dev_bank.jsonl", "rb").read()).hexdigest()
SANITY_HASH = hashlib.sha256(open("bank/sanity/sanity_set.jsonl", "rb").read()).hexdigest()
print(DEV_HASH, SANITY_HASH)

## 1. LLaDA's answer-slot count (1, chosen earlier on the development bank) and no answer prefix

In [ ]:
N = 1
PREFIX = "none"

## 2. Smoke test on a tiny random-weight copy of the official code (CPU)

In [ ]:
!python -m harness.diffusion.checks.check_tiny --model llada --llada-repo /content/LLaDA --answer-prefix {PREFIX} --report {OUT}/tiny.json

## 3. Scoring-code tests 1–3 (PLAN.md §4)
Test 2 compares our scorer with the official model code's first forward pass; test 1 is the sanity bar (AUROC ≥ 0.90); test 3 checks batched against unbatched scores and repeat runs.

In [ ]:
!python -m harness.diffusion.checks.check_parity --model llada --n-slots {N} --answer-prefix {PREFIX} --llada-repo /content/LLaDA --report {OUT}/test2_parity_n{N}.json
!python -m harness.diffusion.checks.check_sanity --model llada --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test1_sanity_n{N}.json --out {OUT}/sanity_scores_n{N}.jsonl
!python -m harness.diffusion.checks.check_invariance --model llada --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test3_invariance_n{N}.json --save {OUT}/invariance_scores_n{N}.json

## 4. Agreement with the saved LLaDA run
Scores the 50 sanity items unbatched with this scorer and compares them with the run in `results/llada-comparison-20261019-213107` (every score within 0.05, Spearman above 0.999).

In [ ]:
!python -m harness.diffusion.run_score --model llada --n-slots 1 --batch-size 1 --items bank/sanity/sanity_set.jsonl --bank-hash {SANITY_HASH} --out {OUT}/sanity_T1_n1.jsonl
!python -m harness.llada_compare --parity {OUT}/sanity_T1_n1.jsonl | tee {OUT}/agreement_with_saved_run.json

Each check prints PASS or FAIL and writes its report even when it fails; a FAIL is a result, not a crash. Keep going.

## Download the results
This zips `results/stage3/llada/` (scores, manifests and reports). Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/stage3_llada.zip results/stage3/llada
from google.colab import files
files.download("/content/stage3_llada.zip")